# Vicuña · V1: variables derivadas del viento del GFS

**Hipótesis.** U y V son cerca del 62 % del error normalizado y V es la peor. Darle a la red la velocidad, el seno y coseno de la dirección, la divergencia y la vorticidad del viento del GFS (calculadas solo con U y V oficiales) mejora U y V.

**Un solo cambio** sobre r2 (ronda 2, dropout 0.1): `derivadas = True`, que agrega 5 canales por hora a la entrada (9 por hora en vez de 4; el codificador pasa de 4·3+2 = 14 a 9·3+2 = 29 canales). Divergencia y vorticidad con diferencias finitas sobre la grilla GFS 17x13 con el espaciado real en metros (fila 0 = norte; las derivadas usan las coordenadas, así el signo sale correcto). El residuo sigue usando el GFS crudo de la hora t. Entrada solo con t-2, t-1 y t (sin horas futuras). Todo lo demás es igual a `vicuna-t21-ronda3` (celdas copiadas tal cual; la celda de `validacion.py` no se toca).

**Base de comparación.** Los 5 modelos de r2 por corte ya entrenados en `vicuna-t21-ronda3` (`resultados_c.json`, semillas 0 a 4). Para comprobar que el código no cambió la base, se reentrena r2 semilla 0 en el corte A: debe quedar a menos de 0.003 de 0.46853 (la GPU no es determinista).

**Regla fijada antes de correr.** V1 (ensamble de 5 semillas por corte) se **acepta** si:
1. reproduce la base (punto anterior),
2. el ensamble V1 supera al ensamble r2 en el corte A (0.48478) **y** en el B (0.48422), y
3. la media de las dos diferencias es >= **0.003** (el mayor entre 0.001 y el ruido entre semillas de r2: 0.0029 en A y 0.0069 en B por modelo).

Se reporta además el RMSE por variable frente a r2 (A: U 1.562, V 1.479; B: U 1.583, V 1.406). Si se acepta, se reentrena con todo el train y se escribe `submission_v1.csv` como candidato. No se envía desde aquí.


In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
def windows(X, t, n=3):
    """Ventana de n horas (t-n+1 .. t). Si falta una hora, se repite la hora t (como en el starter)."""
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), n) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate(range(n - 1, -1, -1)):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (N, n, 17, 13, 4)

WTR = {n: windows(Xtr, ttr, n) for n in (3, 6)}
WTE = {n: windows(Xte, tte, n) for n in (3, 6)}
print({n: (WTR[n].shape, WTE[n].shape) for n in WTR})

In [ ]:
import json, platform
QUICK = False                      # True: prueba corta de que todo corre (en CPU)
if not torch.cuda.is_available():
    print('ADVERTENCIA: sin GPU; solo sirve para QUICK')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EQUIPO = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU'
print(EQUIPO, '·', platform.python_version(), np.__version__, torch.__version__)
# Dos cortes temporales: A (el de T02) y B (1-15 jun, sin mirar nada posterior)
VENTANAS = {'A': dict(usar=np.ones(len(ttr), bool), ini='2024061600'),
            'B': dict(usar=ttr < '2024061600', ini='2024060100')}
for k, v in VENTANAS.items():
    v['trn'] = v['usar'] & (ttr < v['ini']); v['val'] = v['usar'] & (ttr >= v['ini'])
    print(f"ventana {k}: train {v['trn'].sum()} h · val {v['val'].sum()} h · {ttr[v['val']][0]} → {ttr[v['val']][-1]}")
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

In [ ]:
# Grilla WRF dentro de la grilla GFS (como en el starter): GFS fila 0 = norte, WRF y DEM fila 0 = sur
glat, glon = st['gfs_lats'], st['gfs_lons']
wlat, wlon = st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = to_t(np.stack([gx, gy], -1)[None])                      # (1,70,35,2)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

def media_local(a, k):
    """Media en una ventana k x k con borde reflejado."""
    p = np.pad(a, k // 2, mode='reflect')
    return np.lib.stride_tricks.sliding_window_view(p, (k, k)).mean((-1, -2))

def relieve(dem):
    """Pendiente (norte, este, magnitud), orientación (sin, cos) y relieve relativo (~15 y ~45 km)."""
    z = dem.astype(np.float64)
    dn, de = np.gradient(z)                                    # fila crece hacia el norte, columna al este
    mag = np.hypot(dn, de)
    unit = np.where(mag > 0, mag, 1.0)
    capas = [dn, de, mag, de / unit, dn / unit, z - media_local(z, 5), z - media_local(z, 15)]
    return np.stack([(c - c.mean()) / c.std() for c in capas])

TERRENO = to_t(relieve(dem))[None]                             # (1,7,70,35)

def interpolar(a):
    """(B,C,17,13) GFS -> (B,C,70,35) WRF, bilineal."""
    return F.grid_sample(a, GRID.expand(len(a), -1, -1, -1), align_corners=True)

class Bloque(nn.Module):
    def __init__(self, w, d, p=0.0):
        super().__init__()
        self.p = p
        self.f = nn.Sequential(nn.Conv2d(w, w, 3, padding=d, dilation=d), nn.GELU(),
                               nn.Conv2d(w, w, 3, padding=d, dilation=d))
    def forward(self, x):
        return F.gelu(x + F.dropout2d(self.f(x), self.p, self.training))

class Net(nn.Module):
    """Red de la ronda 1 con ancho, bloques, codificador, dropout y número de horas configurables.
    `nv` = variables de entrada por hora (4 sin derivadas, 9 con derivadas)."""
    def __init__(self, skip=False, terreno=False, profundo=False, w=64,
                 bloques=(1, 2, 4, 1), enc_prof=False, dropout=0.0, lags=3, nv=4):
        super().__init__()
        self.skip, self.terreno = skip, terreno
        self.enc = nn.Sequential(nn.Conv2d(nv * lags + 2, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.enc2 = nn.Sequential(Bloque(w, 1, dropout), Bloque(w, 1, dropout)) if enc_prof else nn.Identity()
        c = w + 9 + (nv * lags if skip else 0) + (7 if terreno else 0)
        if profundo:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     *[Bloque(w, d, dropout) for d in bloques], nn.Conv2d(w, 4, 1))
        else:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,lags,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, -1, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        partes = [interpolar(self.enc2(self.enc(torch.cat([x, hh], 1)))), DEM.expand(B, -1, -1, -1),
                  self.static.expand(B, -1, -1, -1)]
        if self.skip:
            partes.append(interpolar(x))
        if self.terreno:
            partes.append(TERRENO.expand(B, -1, -1, -1))
        return self.dec(torch.cat(partes, 1)).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

# ==== V1: variables derivadas del viento del GFS (solo de U y V oficiales) ====
R_TIERRA = 6371000.0
LATR = np.radians(glat.astype(np.float64))                     # (17,), fila 0 = norte (decreciente)
LONR = np.radians(glon.astype(np.float64))                     # (13,), columna 0 = oeste
Y_M = R_TIERRA * LATR                                          # coordenada norte-sur en metros
COSLAT = np.cos(LATR)[:, None]                                 # (17,1)

def viento_derivado(u, v):
    """u, v: (..., 17, 13) en m/s. Devuelve velocidad, cos y sin de la dirección, divergencia y vorticidad (1e-5 s^-1)."""
    u = u.astype(np.float64); v = v.astype(np.float64)
    vel = np.hypot(u, v)
    den = np.sqrt(u * u + v * v + 0.01)                        # epsilon 0.1 m/s para viento calmo
    dudx = np.gradient(u, LONR, axis=-1) / (R_TIERRA * COSLAT)
    dvdx = np.gradient(v, LONR, axis=-1) / (R_TIERRA * COSLAT)
    dudy = np.gradient(u, Y_M, axis=-2)
    dvdy = np.gradient(v, Y_M, axis=-2)
    return np.stack([vel, u / den, v / den, (dudx + dvdy) * 1e5, (dvdx - dudy) * 1e5], -1).astype(np.float32)

def entradas(W, cfg):
    """W: (..., 17, 13, 4) GFS físico (RH, T, U, V). Con derivadas devuelve (..., 17, 13, 9)."""
    W = np.asarray(W, np.float32)
    if not cfg['derivadas']:
        return W
    return np.concatenate([W, viento_derivado(W[..., 2], W[..., 3])], -1)

# Comprobaciones con campos sintéticos: divergencia y vorticidad conocidas, orientación incluida
_x = R_TIERRA * COSLAT * LONR[None, :]                         # (17,13) metros hacia el este
_y = np.broadcast_to(Y_M[:, None], (17, 13))                   # metros hacia el norte
_d = viento_derivado(1e-5 * _x, 1e-5 * _y)                     # u = a x, v = a y  -> div = 2a, vort = 0 en x
assert np.allclose(_d[..., 3], 2.0, atol=1e-6), 'divergencia mal calculada'
_d = viento_derivado(-1e-5 * _y, 0 * _y)                       # u = -a y, v = 0  -> div = 0, vort = a
assert np.allclose(_d[..., 3], 0.0, atol=1e-6) and np.allclose(_d[..., 4], 1.0, atol=1e-6), 'vorticidad mal calculada'
print('derivadas del viento: comprobaciones sintéticas OK')


In [ ]:
BASE = {'w': 64, 'bloques': [1, 2, 4, 1], 'enc_prof': False, 'dropout': 0.0, 'wd': 0.0001, 'lags': 3, 'skip': False, 'residuo': 'crudo', 'terreno': False, 'profundo': True, 'peso_mj': 1, 'epocas': 25, 'derivadas': False}      # configuración final de la ronda 1

def clim_ponderada(A, t, peso):
    """Media por hora UTC en la que las horas de mayo y junio pesan `peso` (1 = climatología de T02)."""
    if peso == 1:
        return climatologia(A, t)
    h = horas(t)
    w = np.where(np.array([s[4:6] in ('05', '06') for s in t]), float(peso), 1.0)
    return np.stack([np.average(A[h == k], axis=0, weights=w[h == k]) for k in range(24)]).astype(np.float32)

def datos_de(mascara, cfg):
    """Normalización y climatologías SOLO con las horas de `mascara`."""
    t = ttr[mascara]
    E = entradas(Xtr[mascara], cfg)
    mu_x = E.mean((0, 1, 2)); sd_x = E.std((0, 1, 2))
    assert (sd_x > 0).all(), 'Hay un canal de entrada constante'
    clim = clim_ponderada(Ytr[mascara], t, cfg['peso_mj'])
    clim_gfs = clim_ponderada(Xtr[mascara], t, cfg['peso_mj'])       # (24,17,13,4)
    anom_sd = (Ytr[mascara] - clim[horas(t)]).std((0, 1, 2))
    D = {'cfg': cfg, 'CLIM': to_t(clim), 'ASD': to_t(anom_sd), 'CLIM_GFS': to_t(clim_gfs),
         'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd, clim_gfs=clim_gfs)}
    return D

def preparar_x(D, W, t):
    """Entrada normalizada, hora y término base del residuo (en anomalías normalizadas de WRF)."""
    s = D['stats']
    x = to_t((entradas(W, D['cfg']) - s['mu_x']) / s['sd_x'])
    h = to_t(horas(t)).long()
    modo = D['cfg']['residuo']
    if modo is None:
        return x, h, None
    partes = []
    for i in range(0, len(W), 256):
        gfs = to_t(W[i:i + 256, -1])                                   # hora t, unidades físicas
        hb = h[i:i + 256]
        if modo == 'anomalia':     # pred = clim_WRF + (GFS - clim_GFS) interpolado + red
            b = interpolar((gfs - D['CLIM_GFS'][hb]).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        elif modo == 'crudo':      # pred = GFS interpolado + red
            b = interpolar(gfs.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][hb]
        else:
            raise ValueError(modo)
        partes.append(b / D['ASD'])
    return x, h, torch.cat(partes)

def salida(model, x, h, b):
    p = model(x, h.float())
    return p if b is None else p + b

def red_de(cfg):
    nv = 9 if cfg['derivadas'] else 4
    return Net(nv=nv, **{k: cfg[k] for k in ('skip', 'terreno', 'profundo', 'w', 'bloques', 'enc_prof', 'dropout', 'lags')}).to(dev)

def entrenar(D, tensores, epocas, semilla, lote=32):
    x, h, b, y = tensores
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = red_de(D['cfg'])
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=D['cfg']['wd'])
    n = len(x); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            i = orden[s:s + lote]
            pred = salida(model, x[i], h[i], None if b is None else b[i])
            mse = ((pred - y[i]) ** 2).mean(dim=(0, 1, 2)) * (D['ASD'] / ESC) ** 2
            loss = torch.sqrt(mse + 1e-8).mean()       # pérdida 'rmse' de T08
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def tensores_de(D, mascara):
    x, h, b = preparar_x(D, WTR[D['cfg']['lags']][mascara], ttr[mascara])
    y = (to_t(Ytr[mascara]) - D['CLIM'][h]) / D['ASD']
    return x, h, b, y

def predecir_lote(model, D, W, t, bs=128):
    x, h, b = preparar_x(D, W, t); model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            s = slice(i, i + bs)
            p = salida(model, x[s], h[s], None if b is None else b[s])
            out.append((D['CLIM'][h[s]] + p * D['ASD']).cpu().numpy())
    P = np.concatenate(out)
    P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P, k):
    v = VENTANAS[k]
    r = validar(P, Ytr[v['usar']], ttr[v['usar']], inicio_val=v['ini'])
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
N_ENS = 2 if QUICK else 5
CONFIGS = {'r2': {**BASE, 'dropout': 0.1}, 'v1': {**BASE, 'dropout': 0.1, 'derivadas': True}}
# Referencia: r2 en vicuna-t21-ronda3 (resultados_c.json), semillas 0 a 4
REF = {'A': dict(ens=0.48478478362646416, solas=[0.46852816367419337, 0.46815997772566553, 0.4696923605927631, 0.4633177105089389, 0.4710605026692637],
                 rmse={'RH': 9.22385025024414, 'T': 0.837085485458374, 'U': 1.561721920967102, 'V': 1.4791918992996216}),
       'B': dict(ens=0.4842208268688488, solas=[0.4471216142748382, 0.46398357984704797, 0.4477607657102063, 0.453355542566268, 0.45647386475149176],
                 rmse={'RH': 9.148894309997559, 'T': 0.9036696553230286, 'U': 1.5826219320297241, 'V': 1.4064702987670898})}
UMBRAL = 0.003
TOL_REPRO = 0.003
if QUICK:                                  # subconjunto chico: solo para ver que el código corre
    _tensores = tensores_de
    def tensores_de(D, mascara):
        m = mascara.copy(); m[np.where(m)[0][64:]] = False
        return _tensores(D, m)
INICIO = time.time()
os.makedirs('salidas_v1', exist_ok=True)

# Formas: 9 variables por hora con derivadas, 29 canales en el codificador
_D = datos_de(VENTANAS['A']['trn'], CONFIGS['v1'])
_x, _h, _b = preparar_x(_D, WTR[3][:4], ttr[:4])
assert tuple(_x.shape) == (4, 3, 17, 13, 9) and tuple(_b.shape) == (4, 70, 35, 4), (_x.shape, _b.shape)
_net = red_de(CONFIGS['v1'])
assert _net.enc[0].in_channels == 29
print('formas OK:', tuple(_x.shape), '-> salida', tuple(_net(_x, _h.float()).shape), flush=True)
del _D, _x, _h, _b, _net

# 1) La base no cambió: r2 semilla 0 en el corte A
v = VENTANAS['A']; cfg = CONFIGS['r2']
D = datos_de(v['trn'], cfg); T = tensores_de(D, v['trn'])
assert T[0].shape[-1] == 4
m0 = medir(predecir_lote(entrenar(D, T, 1 if QUICK else cfg['epocas'], 0), D, WTR[cfg['lags']][v['val']], ttr[v['val']]), 'A')
linea(f'A r2 s0 reproducción ({time.time() - INICIO:.0f}s)', m0)
REPRODUCE = abs(m0['score_oficial'] - REF['A']['solas'][0]) < TOL_REPRO
print('reproduce la base:', REPRODUCE, f"(dif {m0['score_oficial'] - REF['A']['solas'][0]:+.5f})", flush=True)
del D, T; torch.cuda.empty_cache()

# 2) V1: ensamble de N_ENS semillas en cada corte
res = {}
for k, v in VENTANAS.items():
    cfg = CONFIGS['v1']
    D = datos_de(v['trn'], cfg); T = tensores_de(D, v['trn'])
    Wv = WTR[cfg['lags']][v['val']]
    suma, solas = 0, []
    for semilla in range(N_ENS):
        mod = entrenar(D, T, 1 if QUICK else cfg['epocas'], semilla)
        Ps = predecir_lote(mod, D, Wv, ttr[v['val']])
        suma = suma + Ps; solas.append(medir(Ps, k))
        linea(f'{k} v1 s{semilla} ({time.time() - INICIO:.0f}s)', solas[-1])
        torch.save(mod.state_dict(), f'salidas_v1/v1_{k}_s{semilla}.pt')
    P = suma / N_ENS
    res[k] = dict(medir(P, k), solas=[m['score_oficial'] for m in solas], rmse_solas=[m['rmse_3h'] for m in solas])
    linea(f'{k} ensamble v1 ({time.time() - INICIO:.0f}s)', res[k])
    print(f'{k} RMSE v1 - r2:', {x: round(res[k]['rmse_3h'][x] - REF[k]['rmse'][x], 4) for x in VARS}, flush=True)
    np.savez(f'salidas_v1/v1_{k}_val.npz', times=ttr[v['val']], P=P.astype(np.float32))
    np.savez(f'salidas_v1/v1_{k}_estadisticas.npz', **D['stats'])
    del D, T; torch.cuda.empty_cache()

dif = {k: res[k]['score_oficial'] - REF[k]['ens'] for k in VENTANAS}
dif_solas = {k: float(np.mean(np.array(res[k]['solas']) - np.array(REF[k]['solas'][:N_ENS]))) for k in VENTANAS}
dif_rmse = {k: {x: res[k]['rmse_3h'][x] - REF[k]['rmse'][x] for x in VARS} for k in VENTANAS}
media = float(np.mean(list(dif.values())))
ACEPTA = bool(REPRODUCE and media >= UMBRAL and all(d > 0 for d in dif.values()))
print(f'ensamble v1 - r2: {dif} · media {media:+.5f} · por semilla (pareado): {dif_solas} · se acepta: {ACEPTA}', flush=True)
json.dump({'configs': CONFIGS, 'referencia_r2': REF, 'reproduccion_r2_s0_A': m0, 'reproduce': REPRODUCE,
           'resultados': res, 'diferencia': dif, 'diferencia_solas': dif_solas, 'diferencia_rmse': dif_rmse,
           'media': media, 'umbral': UMBRAL, 'acepta': ACEPTA, 'quick': QUICK, 'usa_horas_futuras': False,
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'equipo': EQUIPO}},
          open('resultados_v1.json', 'w'), indent=1)
json.dump(CONFIGS['v1'], open('salidas_v1/v1_config.json', 'w'))


In [ ]:
# Reentrenamiento de V1 con TODOS los datos, solo si se acepta (o en QUICK, para probar el código)
if ACEPTA or QUICK:
    todo = np.ones(len(ttr), bool)
    os.makedirs('models', exist_ok=True)
    sample = pd.read_csv(find('sample_submission.csv'))
    pos = {k: i for i, k in enumerate(tte)}
    parts = sample['id'].str.split('_', expand=True)
    ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
    cfg = CONFIGS['v1']
    D = datos_de(todo, cfg); T = tensores_de(D, todo)
    suma = 0
    for semilla in range(N_ENS):
        mod = entrenar(D, T, 1 if QUICK else cfg['epocas'], semilla)
        suma = suma + predecir_lote(mod, D, WTE[cfg['lags']], tte)
        torch.save(mod.state_dict(), f'models/v1_s{semilla}.pt')
    np.savez('models/estadisticas_v1.npz', **D['stats'])
    json.dump(cfg, open('models/config_v1.json', 'w'))
    P = suma / N_ENS
    sub = pd.DataFrame({'id': sample['id']})
    for k, v in enumerate(VARS):
        sub[v] = P[ti, r, c, k].round(3)
    assert len(sub) == len(sample) and np.isfinite(sub[list(VARS)].values).all()
    sub.to_csv('submission_v1.csv', index=False)
    print('submission_v1.csv', sub.shape, f'· total {time.time() - INICIO:.0f}s')
else:
    print('V1 no se acepta: no se reentrena con todos los datos')
